# Physics Simulation Starter

This notebook sets up a simple Python environment for running basic physics simulations.

In [33]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")  # keep your working backend

# --- Torus parameters ---
R = 0.125   # major radius
r_plasma = 0.03   # plasma minor radius
r_coil   = 0.07   # coil pack minor radius

# --- Create torus meshes ---
plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

# --- Plot both ---
plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_axes()
plotter.show()



Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25810_26&reconnect=auto" class="pyvi…

In [34]:
import numpy as np

# --- Coil parameters ---
N = 200          # turns
I = 2200         # current per turn (A)
NI = N * I       # total ampere-turns

# --- Coil cross-section area ---
r_plasma = 0.03
r_coil   = 0.07

A_coil = np.pi * (r_coil**2 - r_plasma**2)

# --- Current density (A/m^2) ---
J = NI / A_coil

J


35014087.480216965

In [35]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# --- Geometry (reuse from before) ---
R = 0.125
r_plasma = 0.03
r_coil   = 0.07

plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

# --- Sample points on the plasma torus surface ---
points = plasma_torus.points  # (N, 3) array of XYZ

# --- Dummy toroidal B-field: constant magnitude, tangent to torus ---
# For now, we just set a simple vector field that points along +Y everywhere
B = np.tile(np.array([0.0, 0.7, 0.0]), (points.shape[0], 1))  # 0.7 T along +Y

# --- Create a PyVista point cloud with vectors ---
cloud = pv.PolyData(points)
cloud["B"] = B

# --- Plot geometry + field vectors ---
plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")
plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25e50_27&reconnect=auto" class="pyvi…

In [36]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# --- Geometry ---
R = 0.125
r_plasma = 0.03
r_coil   = 0.07

plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

# --- Sample points on the plasma torus ---
points = plasma_torus.points  # (N, 3)

# --- Compute toroidal field direction at each point ---
B0 = 0.7  # Tesla

B = np.zeros_like(points)

for i, (x, y, z) in enumerate(points):
    phi = np.arctan2(y, x)
    # unit vector in toroidal direction
    B[i] = B0 * np.array([-np.sin(phi), np.cos(phi), 0.0])

# --- Attach field to point cloud ---
cloud = pv.PolyData(points)
cloud["B"] = B

# --- Plot ---
plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")
plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25950_28&reconnect=auto" class="pyvi…

In [37]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

# --- Geometry parameters ---
R = 0.125
r_plasma = 0.03
r_coil   = 0.07

# --- Coil parameters ---
N = 200
I = 2200
NI = N * I
A_coil = np.pi * (r_coil**2 - r_plasma**2)
J = NI / A_coil   # current density

# --- Create torus meshes ---
plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

# --- Field solver scaffold ---
def B_field(x, y, z):
    """
    Placeholder magnetic field solver.
    Returns a toroidal field inside the plasma region,
    and zero outside.
    """
    # distance from torus centerline
    r_xy = np.sqrt(x**2 + y**2)
    dist_from_major = np.sqrt((r_xy - R)**2 + z**2)

    # inside plasma region
    if dist_from_major < r_plasma:
        phi = np.arctan2(y, x)
        B0 = 0.7  # Tesla
        return np.array([-np.sin(phi)*B0, np.cos(phi)*B0, 0.0])

    # outside plasma region
    return np.array([0.0, 0.0, 0.0])

# --- Sample grid around torus ---
grid_x = np.linspace(-0.35, 0.35, 40)
grid_y = np.linspace(-0.35, 0.35, 40)
grid_z = np.linspace(-0.20, 0.20, 20)

points = np.array([[x, y, z] for x in grid_x for y in grid_y for z in grid_z])

# --- Compute B-field on grid ---
B = np.array([B_field(x, y, z) for x, y, z in points])

# --- Visualise ---
cloud = pv.PolyData(points)
cloud["B"] = B

plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")
plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b5bba25a90_29&reconnect=auto" class="pyvi…

In [38]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend("trame")

mu0 = 4e-7 * np.pi

# --- Geometry ---
R = 0.125
r_plasma = 0.03
r_coil   = 0.07

# --- Coil parameters ---
N_loops = 20        # reduced for speed
I = 2200
theta_samples = 200

# --- Precompute loops, dl, and midpoints ---
phi_vals = np.linspace(0, 2*np.pi, N_loops, endpoint=False)
theta_vals = np.linspace(0, 2*np.pi, theta_samples)

segments = []

for phi in phi_vals:
    loop = []
    for theta in theta_vals:
        x = (R + r_coil * np.cos(theta)) * np.cos(phi)
        y = (R + r_coil * np.cos(theta)) * np.sin(phi)
        z = r_coil * np.sin(theta)
        loop.append([x, y, z])
    loop = np.array(loop)

    # segment endpoints
    p1 = loop
    p2 = np.roll(loop, -1, axis=0)

    dl = p2 - p1
    mid = 0.5 * (p1 + p2)

    segments.append((p1, dl, mid))

# --- Vectorised Biot–Savart ---
def B_field_fast(x, y, z):
    r = np.array([x, y, z])
    B = np.zeros(3)

    for p1, dl, mid in segments:
        r_vec = r - mid
        dist = np.linalg.norm(r_vec, axis=1)
        mask = dist > 1e-6

        dB = mu0 * I / (4*np.pi) * np.cross(dl[mask], r_vec[mask]) / (dist[mask]**3)[:, None]
        B += dB.sum(axis=0)

    return B

# --- Sample grid ---
grid_x = np.linspace(-0.35, 0.35, 40)
grid_y = np.linspace(-0.35, 0.35, 40)
grid_z = np.linspace(-0.20, 0.20, 20)

points = np.array([[x, y, z] for x in grid_x for y in grid_y for z in grid_z])

# --- Compute B-field ---
B = np.array([B_field_fast(x, y, z) for x, y, z in points])

# --- Visualise ---
cloud = pv.PolyData(points)
cloud["B"] = B

plasma_torus = pv.ParametricTorus(R, r_plasma)
coil_torus   = pv.ParametricTorus(R, r_coil)

plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")
plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b53e91fed0_30&reconnect=auto" class="pyvi…

In [39]:
import numpy as np

# --- Nozzle coil parameters ---
nozzle_radius = 0.02
nozzle_current = 1500
nozzle_theta = np.linspace(0, 2*np.pi, 200)

# Position nozzle coil at φ = 0 (front of torus)
nozzle_center = np.array([R + r_coil + 0.02, 0.0, 0.0])  # slight outward offset

# Compute nozzle coil loop points
nozzle_loop = np.array([
    nozzle_center + np.array([
        nozzle_radius * np.cos(t),
        0.0,
        nozzle_radius * np.sin(t)
    ])
    for t in nozzle_theta
])

# Precompute nozzle segments
nozzle_p1 = nozzle_loop
nozzle_p2 = np.roll(nozzle_loop, -1, axis=0)
nozzle_dl = nozzle_p2 - nozzle_p1
nozzle_mid = 0.5 * (nozzle_p1 + nozzle_p2)

def B_field_nozzle(x, y, z):
    r = np.array([x, y, z])
    B = np.zeros(3)

    r_vec = r - nozzle_mid
    dist = np.linalg.norm(r_vec, axis=1)
    mask = dist > 1e-6

    dB = mu0 * nozzle_current / (4*np.pi) * np.cross(nozzle_dl[mask], r_vec[mask]) / (dist[mask]**3)[:, None]
    B += dB.sum(axis=0)

    return B

# --- Combined field ---
def B_total(x, y, z):
    return B_field_fast(x, y, z) + B_field_nozzle(x, y, z)

# --- Compute combined field on grid ---
B_combined = np.array([B_total(x, y, z) for x, y, z in points])

# --- Visualise ---
cloud = pv.PolyData(points)
cloud["B"] = B_combined

plotter = pv.Plotter()
plotter.add_mesh(plasma_torus, color="cyan", opacity=0.6)
plotter.add_mesh(coil_torus, color="orange", opacity=0.3)
plotter.add_mesh(cloud.glyph(orient="B", scale="B", factor=0.08), color="red")

# Visualise nozzle coil
plotter.add_mesh(pv.PolyData(nozzle_loop), color="green", point_size=5)

plotter.add_axes()
plotter.show()


Widget(value='<iframe src="http://localhost:60710/index.html?ui=P_0x2b53e727b10_31&reconnect=auto" class="pyvi…